# Naive Bayes Classifier for Bank Fraud Detection
This notebook trains a Naive Bayes model on cleaned fraud data and evaluates it using standardized metrics.

## 1. Imports and Setup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, precision_score, recall_score, average_precision_score, 
                             roc_auc_score, precision_recall_curve, f1_score, make_scorer)
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

## 2. Evaluation Function

In [3]:
def evaluate_model(y_true, y_pred, y_proba=None, plot_pr_curve=True):
    print("Classification Report:")
    print(classification_report(y_true, y_pred, digits=4))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix:")
    print(cm)

    # Key Metrics
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"\nPrecision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")

    metrics = {
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'confusion_matrix': cm.tolist()
    }

    # Probabilistic metrics
    if y_proba is not None:
        pr_auc = average_precision_score(y_true, y_proba)
        metrics['auc_pr'] = pr_auc
        print(f"AUC-PR (Average Precision): {pr_auc:.4f}")

        roc_auc = roc_auc_score(y_true, y_proba)
        metrics['roc_auc'] = roc_auc
        print(f"ROC AUC:                   {roc_auc:.4f}")

        if plot_pr_curve:
            precision_pts, recall_pts, _ = precision_recall_curve(y_true, y_proba)
            plt.figure(figsize=(6, 5))
            plt.plot(recall_pts, precision_pts, marker='.', label='PR Curve')
            plt.xlabel('Recall')
            plt.ylabel('Precision')
            plt.title(f'Precision-Recall Curve (AUC = {pr_auc:.4f})')
            plt.grid(True)
            plt.legend()
            plt.tight_layout()
            plt.show()

    return metrics

## 3. Load and Prepare Data

In [4]:
folds = []

for i in range(1, 6):
    val_df = pd.read_csv(f"../../data/preprocessed/fold_{i}_val.csv")
    train_dfs = []

    for j in range(1, 6):
        if j != i:
            train_dfs.append(pd.read_csv(f"../../data/preprocessed/fold_{j}_train.csv"))
    
    train_df = pd.concat(train_dfs, ignore_index=True)
    folds.append((train_df, val_df))

## 4. Train Naive Bayes Model with Hyperparameter Tuning & 5. Evaluate Model

In [ ]:
param_grid = {
    'var_smoothing': np.logspace(-12, -6, 7)
}

for fold_idx, (train_df, val_df) in enumerate(folds):
    print(f"\n--- Fold {fold_idx + 1} ---")

    # Split features and labels
    X_train = train_df.drop(columns='fraud_bool')
    y_train = train_df['fraud_bool']

    X_val = val_df.drop(columns='fraud_bool')
    y_val = val_df['fraud_bool']

    # Feature Selection using RandomForest
    selector_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    selector_model.fit(X_train, y_train)
    selector = SelectFromModel(selector_model, threshold='median', prefit=True)

    X_train_selected = selector.transform(X_train)
    X_val_selected = selector.transform(X_val)

    # Parameter tuning with GridSearch / Initialize and perform GridSearch
    gnb = GaussianNB()
    grid_search_nb = GridSearchCV(estimator=gnb,
                                   param_grid=param_grid,
                                   scoring='f1',
                                   cv=5,
                                   verbose=1,
                                   n_jobs=1)

    grid_search_nb.fit(X_train_selected, y_train)

    print("Best Parameters:", grid_search_nb.best_params_)
    print("Best F1 Score from Inner CV:", grid_search_nb.best_score_)

    # Predict and evaluate on the validation set
    best_model = grid_search_nb.best_estimator_
    y_pred = best_model.predict(X_val_selected)
    y_proba = best_model.predict_proba(X_val_selected)[:, 1]

    evaluate_model(y_val, y_pred, y_proba)


--- Fold 1 ---


KeyboardInterrupt: 